# 1. Data quality and the research question
Can a historical manufacturing risk model remain useful in later periods? This is an AI-assisted, independent MSc portfolio study using public anonymous data, not a course submission or industrial engagement. All periods were previously inspected; later results are retrospective. Independent author mastery is not established by these executed cells.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
OUT = ROOT / 'reports/study'
summary = json.loads((OUT / 'summary.json').read_text())
def table(name): return pd.read_csv(OUT / f'{name}.csv')


## Audit the actual input
Labels represent recorded pass/fail outcomes. Sensor identity, units, equipment and specifications are unavailable. Anonymous variables cannot be assigned a physical mechanism.

In [2]:
from secom_project.data import load_secom
X, y, timestamps = load_secom(ROOT / 'data/raw')
display(pd.Series({'rows': len(X), 'variables': X.shape[1], 'failures': int(y.sum()), 'missing_cells': int(X.isna().sum().sum()), 'duplicate_timestamps': int(timestamps.duplicated().sum())}).to_frame('value'))
assert len(X) == summary['data']['rows']
display(table('data_quality').sort_values('missing_rate', ascending=False).head(10))

,value
rows,1567
variables,590
failures,104
missing_cells,41951
duplicate_timestamps,33


,feature,missing_rate,observed_unique_values
292,sensor_292,0.911934,92
293,sensor_293,0.911934,138
158,sensor_158,0.911934,138
157,sensor_157,0.911934,128
492,sensor_492,0.855775,226
85,sensor_085,0.855775,97
358,sensor_358,0.855775,20
220,sensor_220,0.855775,69
244,sensor_244,0.649649,66
517,sensor_517,0.649649,543


## Why time is part of the question
Failure proportions and sample sizes change across time. This does not by itself prove concept drift or a physical cause. Equal timestamps are grouped at each split; unknown lot dependence remains unresolved.

In [3]:
weekly = table('weekly_failure_rate')
display(weekly)
display(table('splits'))

,timestamp,size,sum,mean
0,2008-07-19,24,5,0.208333
1,2008-07-26,47,10,0.212766
2,2008-08-02,93,7,0.075269
3,2008-08-09,48,7,0.145833
4,2008-08-16,217,27,0.124424
5,2008-08-23,122,5,0.040984
6,2008-08-30,185,6,0.032432
7,2008-09-06,84,2,0.023810
8,2008-09-13,112,7,0.062500
9,2008-09-20,184,2,0.010870


,window,train_rows,evaluation_rows,train_start,train_end,evaluation_start,evaluation_end,train_failures,evaluation_failures
0,1,783,157,2008-07-19 11:55:00,2008-09-11 07:43:00,2008-09-11 08:06:00,2008-09-20 05:34:00,67,9
1,2,940,156,2008-07-19 11:55:00,2008-09-20 05:34:00,2008-09-20 06:08:00,2008-09-26 02:26:00,76,2
2,3,1096,157,2008-07-19 11:55:00,2008-09-26 02:26:00,2008-09-26 03:12:00,2008-10-02 19:25:00,78,9
3,4,1253,314,2008-07-19 11:55:00,2008-10-02 19:25:00,2008-10-02 20:54:00,2008-10-17 06:07:00,87,17


## Provenance
The manifest records raw-data hashes, source hashes, package versions and the actual configuration. Its git HEAD is the base revision at generation; source hashes identify any uncommitted implementation used.

In [4]:
import hashlib
manifest = json.loads((OUT / 'manifest.json').read_text())
for name, expected in manifest['data_sha256'].items():
    assert hashlib.sha256((ROOT/'data/raw'/name).read_bytes()).hexdigest() == expected
display(pd.Series(manifest['package_versions']).to_frame('version'))

,version
numpy,2.5.2
pandas,2.3.3
scikit-learn,1.9.0
scipy,1.18.1
matplotlib,3.11.1
seaborn,0.13.2
streamlit,1.63.0
nbformat,5.11.1
nbclient,0.11.0
pytest,8.4.2
